In [1]:
import os
import keras.losses
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks
from copy import deepcopy

In [2]:
# --- データ定義 ---
data_dir = "C:/Users/takat/Desktop/sotuken/dat_to_images_64"

#辞書にしとけ
# ラベルがついているPBMファイル
label_dict = {
    # Mild (0)
    'AF483470.1.pbm': 0, 'EF192393.1.pbm': 0, 'EF192394.1.pbm': 0,
    'EF580923.1.pbm': 0, 'EU879915.1.pbm': 0, 'EU879916.1.pbm': 0,
    'JQ806338.1.pbm': 0, 'KF418767.1.pbm': 0, 'KR611355.1.pbm': 0,
    'KT987925.1.pbm': 0, 'LC388852.1.pbm': 0, 'LC388854.1.pbm': 0,
    'M25199.1.pbm': 0, 'MG450357.1.pbm': 0, 'Y09575.1.pbm': 0,

    # Moderate (1)
    'AF454395.1.pbm': 1, 'KF683200.1.pbm': 1, 'KJ857496.1.pbm': 1,
    'KR611360.1.pbm': 1, 'M88678.1.pbm': 1, 'X17268.1.pbm': 1,
    'GQ853461.1.pbm': 1, 'EU879913.1.pbm': 1,

    # Severe (2)
    'AJ634596.1.pbm': 2, 'AY518939.1.pbm': 2, 'AY532801.1.pbm': 2,
    'DD220185.1.pbm': 2, 'FR851463.1.pbm': 2, 'JX280944.1.pbm': 2,
    'U23060.1.pbm': 2, 'X58388.1.pbm': 2, 'X76846.1.pbm': 2,
    'X97387.1.pbm': 2, 'Y09383.1.pbm': 2, 'LC523672.1.pbm': 2,
    'LC523675.1.pbm': 2, 'LC523676.1.pbm': 2
}

# --- 既存変数形式に変換しておく（下位互換のため） ---
labeled_pbm = list(label_dict.keys())
labels = list(label_dict.values())

# --- 全ファイルを取得 ---
filepaths = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".pbm")]

# --- ラベルリストを生成（未ラベルは -1） ---
labels_list = [label_dict.get(os.path.basename(fp), -1) for fp in filepaths]

# --- その他設定 ---
IMG_HEIGHT = 64
IMG_WIDTH = 64
BATCH_SIZE = 1
num_classes = 3
label_map = {0: "mild", 1: "moderate", 2: "severe"}

print(f"ラベル付きデータ数: {len(labeled_pbm)}")
print(f"全ファイル数: {len(filepaths)}")
print(f"未ラベルファイル数: {labels_list.count(-1)}")


ラベル付きデータ数: 37
全ファイル数: 307
未ラベルファイル数: 270


In [3]:
def load_and_preprocess_list(fp_list):
    X = []
    for p in fp_list:
        img = np.array(Image.open(p).convert('L').resize((64, 64)), dtype=np.float32)
        img = np.expand_dims(img, axis=-1)
        X.append(img)
    return np.array(X)

In [4]:
def load_and_preprocess_list_gabege(fp_list):
    X = []
    for p in fp_list:
        img = np.array(Image.open(p))
        img = img.reshape(64, 64, 1)
        img = img.astype('float32')
        img /= 1.0
        X.append(img)
    return np.array(X)

In [5]:
def make_model():
    model = models.Sequential()
    model.add(layers.Conv2D(16,(3,3), activation='relu', input_shape=(64, 64, 1)))
    model.add(layers.MaxPool2D((2,2)))
    model.add(layers.Conv2D(32,(3,3), activation='relu'))
    model.add(layers.MaxPool2D((2,2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.1))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model


In [6]:
def jackknife_on_files(labeled_paths, labeled_labels, unlabeled_paths, epochs, verbose=1):
    n = len(labeled_paths)
    X_labeled = load_and_preprocess_list(labeled_paths)
    y_labeled = np.array(labeled_labels, dtype=np.int32)
    X_unlabeled = load_and_preprocess_list(unlabeled_paths)
    val_accs = []
    val_records = []  # ← 各foldの情報を格納するリストを追加

    all_preds = {os.path.basename(fp): {"True_Label": label_map.get(lb, "")} for fp, lb in zip(labeled_paths + unlabeled_paths, labels_list)}

    for i in range(n):
        print(f"\n=== Fold {i+1}/{n} ===")
        mask = np.arange(n) != i
        x_train, y_train = X_labeled[mask], y_labeled[mask]
        x_val, y_val = X_labeled[[i]], y_labeled[[i]]

        val_name = os.path.basename(labeled_paths[i])              # 検証ファイル名
        val_true_label = label_map[int(y_val[0])]                  # 検証データの真のラベル

        model = make_model()
        model.compile(optimizer=optimizers.Adam(1e-5),
                      loss=losses.SparseCategoricalCrossentropy(),
                      metrics=['accuracy'])

        history = model.fit(x_train, y_train, epochs=epochs, verbose=verbose, validation_data=(x_val, y_val))
        val_acc = np.mean(history.history["val_accuracy"])
        val_accs.append(val_acc)

        print(f"Mean Val Accuracy (Fold {i+1}): {val_acc:.4f}")

        # 検証データ情報を記録
        val_records.append({
            "Fold": i + 1,
            "Val_Filename": val_name,
            "Val_True_Label": val_true_label,
            "Mean_Val_Acc": val_acc
        })

        all_data = labeled_paths + unlabeled_paths
        all_X = np.concatenate([X_labeled, X_unlabeled], axis=0)
        preds = np.argmax(model.predict(all_X, verbose=0), axis=1)

        for path, pred in zip(all_data, preds):
            name = os.path.basename(path)
            all_preds[name][f"Fold{i+1}_Pred"] = label_map[int(pred)]

    # --- DataFrame化 ---
    df = pd.DataFrame.from_dict(all_preds, orient="index")
    df.reset_index(inplace=True)
    df.rename(columns={"index": "Filename"}, inplace=True)

    df_val = pd.DataFrame(val_records)  # ← foldごとのval情報をDataFrameにまとめる

    return df, df_val


In [ ]:
labeled_fullpaths = [os.path.join(data_dir, name) for name in labeled_pbm]
unlabeled_fullpaths = [fp for fp, lb in zip(filepaths, labels_list) if lb == -1]

df_pred, df_acc = jackknife_on_files(labeled_fullpaths, labels, unlabeled_fullpaths, epochs=100, verbose=1)

df_pred.to_csv("jackknife_all_predictions.csv", index=False, encoding="utf-8-sig")
df_acc.to_csv("jackknife_val_accuracy.csv", index=False, encoding="utf-8-sig")

print("finish")


=== Fold 1/37 ===


C:\Users\takat\Desktop\sotuken\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 283ms/step - accuracy: 0.1944 - loss: 15.1866 - val_accuracy: 0.0000e+00 - val_loss: 19.6539
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.3333 - loss: 13.0217 - val_accuracy: 0.0000e+00 - val_loss: 15.8515
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.3333 - loss: 12.0212 - val_accuracy: 0.0000e+00 - val_loss: 12.4487
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.4444 - loss: 9.2736 - val_accuracy: 0.0000e+00 - val_loss: 9.2175
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.2778 - loss: 8.3890 - val_accuracy: 0.0000e+00 - val_loss: 6.0198
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3611 - loss: 8.9596 - val_accuracy: 0.0000e+00 - val_loss: 2.8522
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4167 - loss: 6.5160 - val_accuracy: 1.0000 - val_loss: 0.5163
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4722 - loss: 7.9987 - v